# 02 — Mean Reversion

Two contrarian strategies on SPY: RSI(2) Connors entry and Bollinger band fade.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.backtest.runner import run
from src.data.store import get_or_fetch_panel


In [ ]:
from src.strategies.rsi2 import Rsi2
from src.strategies.bollinger import BollingerFade

close = get_or_fetch_panel(['SPY'], '2010-01-01')
close.tail(3)


## RSI(2) Connors


In [ ]:
strat = Rsi2(period=2, entry=5.0, exit=70.0)
result = run(close, strat)
print(result.summary())


In [ ]:
pf = result.portfolio
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
pf.value().plot(ax=axes[0]); axes[0].set_title('Equity')
pf.drawdown().plot(ax=axes[1], color='red'); axes[1].set_title('Drawdown')
plt.tight_layout(); plt.show()


In [ ]:
from src.strategies.rsi2 import _rsi
rsi = _rsi(close, 2)
fig, ax = plt.subplots(figsize=(11, 3))
rsi['SPY'].hist(bins=50, ax=ax)
ax.axvline(5, color='g', ls='--', label='entry'); ax.axvline(70, color='r', ls='--', label='exit')
ax.set_title('RSI(2) distribution'); ax.legend(); plt.show()


## Bollinger fade


In [ ]:
strat = BollingerFade(period=20, n_std=2.0)
result = run(close, strat)
print(result.summary())


In [ ]:
pf = result.portfolio
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
pf.value().plot(ax=axes[0]); axes[0].set_title('Equity')
pf.drawdown().plot(ax=axes[1], color='red'); axes[1].set_title('Drawdown')
plt.tight_layout(); plt.show()


In [ ]:
mid = close['SPY'].rolling(20).mean()
std = close['SPY'].rolling(20).std()
fig, ax = plt.subplots(figsize=(11, 4))
tail = close.iloc[-300:]
tail['SPY'].plot(ax=ax, label='SPY')
(mid - 2*std).iloc[-300:].plot(ax=ax, color='red', alpha=0.5, label='lower')
(mid + 2*std).iloc[-300:].plot(ax=ax, color='red', alpha=0.5, label='upper')
mid.iloc[-300:].plot(ax=ax, color='gray', alpha=0.5, label='mid')
ax.legend(); ax.set_title('Bollinger bands (last 300 bars)'); plt.show()
